# Unbounded Graph Expansion — shakedzy/tic_tac_toe

**Smell (`dqn.py` line 171):** Inside the training loop, `model.predict()` and `model.fit()` are called in each iteration without resetting the TF graph. Each call to these Keras methods appends new ops to the default graph — the graph grows unboundedly across training steps, consuming increasing memory and slowing down each subsequent step.

**Fix:** Wrap each training run in `tf.compat.v1.reset_default_graph()` / `K.clear_session()` boundaries and use `@tf.function` to compile the training step once rather than re-tracing it every call.

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings, random
from collections import deque
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS       = 10
N_EPISODES   = 200   # self-play episodes
GAMMA        = 0.95
EPSILON      = 1.0
EPSILON_MIN  = 0.01
EPSILON_DECAY= 0.995
LR           = 0.001
BATCH_SIZE   = 32
MEMORY_SIZE  = 2000
STATE_SIZE   = 9   # 3x3 board
ACTION_SIZE  = 9
print('Config ready')

In [ ]:
# Tic-tac-toe environment
class TicTacToe:
    def __init__(self):
        self.board = np.zeros(9, dtype=np.float32)
        self.done  = False
        self.winner = 0

    def reset(self):
        self.board  = np.zeros(9, dtype=np.float32)
        self.done   = False
        self.winner = 0
        return self.board.copy()

    def available(self):
        return [i for i in range(9) if self.board[i] == 0]

    def step(self, action, player):
        if self.board[action] != 0:
            return self.board.copy(), -10.0, True
        self.board[action] = player
        win = self._check_win(player)
        if win:
            self.done = True; self.winner = player
            return self.board.copy(), 1.0, True
        if len(self.available()) == 0:
            self.done = True
            return self.board.copy(), 0.5, True
        return self.board.copy(), 0.0, False

    def _check_win(self, p):
        b = self.board.reshape(3,3)
        lines = list(b) + list(b.T) + [b.diagonal(), np.fliplr(b).diagonal()]
        return any(np.all(l == p) for l in lines)

def build_dqn():
    model = Sequential([
        Dense(64, input_dim=STATE_SIZE, activation='relu'),
        Dense(64, activation='relu'),
        Dense(ACTION_SIZE, activation='linear'),
    ])
    model.compile(loss='mse', optimizer=Adam(LR))
    return model

print('Environment and model builder ready')

## BEFORE — Smell Active (model.predict/fit called in raw loop — graph grows unboundedly)

In [ ]:
results_before = []

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()

    model   = build_dqn()
    memory  = deque(maxlen=MEMORY_SIZE)
    epsilon = EPSILON
    env     = TicTacToe()

    tracker = EmissionsTracker(
        project_name=f'TicTacToe_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    total_reward = 0.0
    for ep in range(N_EPISODES):
        state = env.reset()
        ep_reward = 0.0
        while True:
            avail = env.available()
            if random.random() < epsilon or not memory:
                action = random.choice(avail)
            else:
                # ❌ SMELL (mirrors dqn.py line 171): model.predict called in loop
                # — each call re-traces and appends ops to the TF graph
                q = model.predict(state.reshape(1,-1), verbose=0)[0]
                mask = np.full(ACTION_SIZE, -np.inf)
                mask[avail] = q[avail]
                action = int(np.argmax(mask))

            next_state, reward, done = env.step(action, 1)
            memory.append((state, action, reward, next_state, done))
            state = next_state
            ep_reward += reward

            if len(memory) >= BATCH_SIZE:
                batch = random.sample(memory, BATCH_SIZE)
                ss  = np.array([b[0] for b in batch])
                aa  = np.array([b[1] for b in batch])
                rr  = np.array([b[2] for b in batch])
                ns  = np.array([b[3] for b in batch])
                dd  = np.array([b[4] for b in batch])
                # ❌ SMELL: model.predict inside loop — unbounded graph nodes
                q_next = model.predict(ns, verbose=0)
                target = rr + GAMMA * np.max(q_next, axis=1) * (~dd)
                q_curr = model.predict(ss, verbose=0)
                q_curr[np.arange(BATCH_SIZE), aa] = target
                # ❌ SMELL: model.fit inside loop — graph grows each step
                model.fit(ss, q_curr, epochs=1, verbose=0)

            if done:
                break

        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        total_reward += ep_reward

    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'avg_reward': round(total_reward / N_EPISODES, 4),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  avg_reward={total_reward/N_EPISODES:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model, memory, env; gc.collect()

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/shakedzy_tic_tac_toe_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/shakedzy_tic_tac_toe_before.csv')

## AFTER — Smell Fixed (compiled train_step with @tf.function — graph traced once)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    K.clear_session(); gc.collect()

    model   = build_dqn()
    memory  = deque(maxlen=MEMORY_SIZE)
    epsilon = EPSILON
    env     = TicTacToe()
    opt     = Adam(LR)
    loss_fn = tf.keras.losses.MeanSquaredError()

    # ✅ FIX: compile inference and update steps as tf.functions — traced once,
    # reused every call without adding new graph nodes
    @tf.function
    def predict_fn(x):
        return model(x, training=False)

    @tf.function
    def train_step(states, targets):
        with tf.GradientTape() as tape:
            preds = model(states, training=True)
            loss  = loss_fn(targets, preds)
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    tracker = EmissionsTracker(
        project_name=f'TicTacToe_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()

    total_reward = 0.0
    for ep in range(N_EPISODES):
        state = env.reset()
        ep_reward = 0.0
        while True:
            avail = env.available()
            if random.random() < epsilon or not memory:
                action = random.choice(avail)
            else:
                q = predict_fn(tf.constant(state.reshape(1,-1))).numpy()[0]
                mask = np.full(ACTION_SIZE, -np.inf)
                mask[avail] = q[avail]
                action = int(np.argmax(mask))

            next_state, reward, done = env.step(action, 1)
            memory.append((state, action, reward, next_state, done))
            state = next_state
            ep_reward += reward

            if len(memory) >= BATCH_SIZE:
                batch = random.sample(memory, BATCH_SIZE)
                ss  = tf.constant(np.array([b[0] for b in batch]))
                aa  = np.array([b[1] for b in batch])
                rr  = np.array([b[2] for b in batch], dtype=np.float32)
                ns  = tf.constant(np.array([b[3] for b in batch]))
                dd  = np.array([b[4] for b in batch])
                q_next = predict_fn(ns).numpy()
                target_vals = rr + GAMMA * np.max(q_next, axis=1) * (~dd)
                q_curr = predict_fn(ss).numpy()
                q_curr[np.arange(BATCH_SIZE), aa] = target_vals
                train_step(ss, tf.constant(q_curr))

            if done:
                break

        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        total_reward += ep_reward

    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'avg_reward': round(total_reward / N_EPISODES, 4),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  avg_reward={total_reward/N_EPISODES:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    del model, memory, env, opt, predict_fn, train_step; gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/shakedzy_tic_tac_toe_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/shakedzy_tic_tac_toe_after.csv')

In [ ]:
print('\n=== SUMMARY: shakedzy/tic_tac_toe ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")